### 🧩 Problem Statement
- **What problem is being solved?**
  Machine Learning models deployed in production can fail silently if the input data changes (Data Drift) or the real-world logic changes (Concept Drift). We need a system to detect this.
- **Why it matters**:
  If a bank's loan model ignores inflation (Income varies), it might reject good customers. If it ignores policy changes, it might approve bad loans.
- **Real-world relevance**:
  Every MLOps pipeline (Netflix, Uber, Banks) has a monitoring layer to trigger retraining when data shifts.

### 🪜 Steps to Solve the Problem
1.  **Generate Synthetic Data**: Create 3 batches (Baseline, Data Drift, Concept Drift).
2.  **Define Checks**: Implement Data Quality (Nulls) and Drift Checks (KS Test).
3.  **Run Pipeline**: Pass data through checks.
4.  **Alert**: If checks fail, print a clear alert.

### 🎯 Expected Output (OVERALL)
- A robust pipeline that prints "✅ SYSTEM HEALTHY" for normal data.
- Prints "🚨 ALERT FIRED" when input distribution shifts (Drift).

### 🔹 Line Explanation
#### 2.1 What the line does
Importing necessary libraries: `numpy` (math), `pandas` (dataframes), `scipy.stats` (statistical tests).

#### 2.2 Why it is used
- **Pandas**: To handle tabular data (like Excel sheets).
- **SciPy**: To run the Kolmogorov-Smirnov (KS) Test for drift detection.

#### 2.3 When to use it
At the start of every Data Science project.

#### 2.4 Where to use it
First cell of the notebook.

#### 2.5 How to use it
Standard Python import syntax.

#### 2.6 How it works internally
Loads the compiled C/Python code for these libraries into memory.

#### 2.7 Output
No visible output, but functions become available.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt

# Set seed for reproducibility
np.random.seed(42)

### ⚙️ Function Explanation: `generate_data`
This function simulates our "Real World" scenarios.

#### Arguments Explanation
- **batch_type (str)**:
    - *What*: Specifies which scenario to generate ('baseline', 'data_drift', 'concept_drift').
    - *Why*: allows us to test different conditions with one function.
    - *Default*: 'baseline'.
- **n_samples (int)**:
    - *What*: How many rows of data to generate.
    - *Why*: More samples = more stable statistical tests.
    - *Default*: 1000.

#### Logic
- **Baseline**: Income Mean = 50k. Logic = Normal.
- **Data Drift**: Income Mean = 75k (Inflation). Logic = Normal.
- **Concept Drift**: Income Mean = 50k (Normal). Logic = Stricter (Harder to get loan).

In [ ]:
def generate_data(batch_type='baseline', n_samples=1000):
    # Base Income Generation
    if batch_type == 'data_drift':
        # DATA DRIFT: Income shifts HIGHER (Mean=75000)
        income = np.random.normal(75000, 15000, n_samples)
    else:
        # NORMAL / CONCEPT DRIFT: Income is Standard (Mean=50000)
        income = np.random.normal(50000, 10000, n_samples)
    
    # Target Generation (Approval Logic)
    if batch_type == 'concept_drift':
        # CONCEPT DRIFT: Logic changes! Harder to get approved (Threshold ~65000)
        prob = 1 / (1 + np.exp(-(income - 65000) / 10000))
    else:
        # BASELINE / DATA DRIFT: Normal Logic (Threshold ~50000)
        prob = 1 / (1 + np.exp(-(income - 50000) / 10000))
        
    approval = (np.random.rand(n_samples) < prob).astype(int)
    
    return pd.DataFrame({'income': income, 'loan_approved': approval})

### 🔹 Line Explanation
#### 2.1 What the line does
Defines `check_data_quality` to verify the health of the incoming dataframe.

#### 2.2 Why it is used
Garbage In, Garbage Out. If data is broken (nulls), the model fails.

#### 2.3 Check Details
1.  **Null Check**: `df.isnull().sum()`. Returns count of missing values. Should be 0.
2.  **Range Check**: `df['income'].min()`. Income should not be negative.

#### 2.6 How it works internally
Pandas scans the C-arrays backing the dataframe to count NaNs or find minimums.

In [ ]:
def check_data_quality(df):
    results = {}
    
    # Check 1: Null Check
    null_count = df.isnull().sum().sum()
    results['null_check_pass'] = (null_count == 0)
    
    # Check 2: Range Check
    min_income = df['income'].min()
    results['range_check_pass'] = (min_income >= 0)
    
    return results

### ⚙️ Function Explanation: `check_drift`
Implements the statistical drift detection.

#### Arguments
- **baseline_df**: The reference data (Training data).
- **current_df**: The new production data (Batch 1 or 2).

#### Key Concept: KS Test (`ks_2samp`)
- **What**: Compares two samples to see if they come from the same distribution.
- **Output**: `p_value`.
    - If **p < 0.05**: Small probability they are same -> **Drift Detected**.
    - If **p >= 0.05**: High probability they are same -> **No Drift**.

#### Key Concept: Mean Check
- **What**: Compares simple average.
- **Rule**: If change > 20%, flag it.

In [ ]:
def check_drift(baseline_df, current_df):
    results = {}
    
    # 1. KS Test (Distribution Check)
    stat, p_value = ks_2samp(baseline_df['income'], current_df['income'])
    results['ks_p_value'] = p_value
    results['drift_detected_ks'] = (p_value < 0.05)
    
    # 2. Mean Shift Check
    mean_base = baseline_df['income'].mean()
    mean_curr = current_df['income'].mean()
    perc_diff = abs(mean_curr - mean_base) / mean_base
    results['drift_detected_mean'] = (perc_diff > 0.20)
    
    return results

### 🔹 Line Explanation
#### 2.1 What the line does
Orchestrates the entire flow: Quality Check -> Drift Check -> Alert.

#### 2.2 Why it is used
To automate the process. We don't want to run checks manually every day.

#### 2.5 How it uses the logic
- If Quality Fails -> Stop immediately.
- If Drift Detected -> Print "🚨 ALERT FIRED".
- Else -> Print "✅ SYSTEM HEALTHY".

In [ ]:
def run_monitoring_pipeline(batch_name, baseline_df, current_df):
    print(f"\n--- MONITORING REPORT: {batch_name} ---")
    
    # 1. Quality
    dq = check_data_quality(current_df)
    if not (dq['null_check_pass'] and dq['range_check_pass']):
        print(f"[CRITICAL] Data Quality Failed.")
        return

    # 2. Drift
    drift = check_drift(baseline_df, current_df)
    
    # 3. Alert Rule
    if drift['drift_detected_ks'] or drift['drift_detected_mean']:
        print(">>> 🚨 ALERT FIRED 🚨 <<<")
        print(f"Reason: KS p-value={drift['ks_p_value']:.5f}")
        print("ACTION: Verify data source health OR Trigger Retraining.")
    else:
        print(">>> ✅ SYSTEM HEALTHY ✅ <<<")

### 📌 Sample Example
We generated 3 batches and now we run the pipeline on them.

### 📊 Expected Output
- **Baseline**: Passes checks.
- **Batch 1 (Data Drift)**: Should ALERT (Income changed).
- **Batch 2 (Concept Drift)**: Should PASS (False Negative), because Income ($X$) didn't change, only Logic ($Y|X$) did.

In [ ]:
# 1. Generate Data
df_baseline = generate_data('baseline')
df_batch1 = generate_data('data_drift')
df_batch2 = generate_data('concept_drift')

# 2. Run Pipeline
run_monitoring_pipeline("Baseline Test", df_baseline, df_baseline)
run_monitoring_pipeline("Batch 1 (Data Drift)", df_baseline, df_batch1)
run_monitoring_pipeline("Batch 2 (Concept Drift)", df_baseline, df_batch2)

### 💼 Interview Perspective
- **Q**: Why did Batch 2 pass the Drift Check even though logic changed?
- **A**: Because **KS Test (Data Drift)** only checks inputs ($X$). It does not see labels ($Y$). To catch Concept Drift, you need **Performance Monitoring** (Accuracy/F1).